In [ ]:
from pyproj import Transformer

transformer = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

In [ ]:
import json
from pyproj import Transformer

transformer = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

def convert_coords(coords):
    if isinstance(coords[0], float):
        x, y = coords
        lon, lat = transformer.transform(x, y)
        return [lon, lat]
    return [convert_coords(c) for c in coords]

def clean_feature(f):
    prop = f.get("properties", {})
    pop = prop.get("population")

    geom = f.get("geometry", {})
    coords = geom.get("coordinates", [])

    new_coords = convert_coords(coords)

    return {
        "type": "Feature",
        "properties": {
            "population": pop
        },
        "geometry": {
            "type": geom.get("type"), 
            "coordinates": new_coords
        }
    }

def convert_to_geojson(data):
    return {
        "type": "FeatureCollection",
        "features": [clean_feature(f) for f in data["features"]]
    }


In [1]:
cities = [
    "Rome",
    "Milan",
    "Munchen",
    "Berlin",
    "Hamburg",
    "Prague",
    "Budapest",
    "Paris",
    "Marseille",
    "Lille",
    "Lyon",
    "Barcelona",
    "Madrid",
    "Sevilla",
    "Bilbao",
    "Lisbon",
    "Porto",
    "Stockholm",
    "Oslo",
    "Copenhagen",
    "Helsinki",
    "Amsterdam",
    "Rotterdam",
    "Bruxelles",
    "Monaco",
    "Sofia",
    "London",
    "Manchester",
    "Glasgow",
    "Dublin",
    "Bucarest",
    "Riga",
    "Tallin",
    "Vilnius",
    "Ljubjiana",
    "Wien",
    "Berna",
    "Zurich",
    "Belgrade",
    "Warsaw"
]

In [3]:
#città
for c in cities:
    with open(c + ".json", "r", encoding="utf-8") as f:
        data = json.load(f)
    converted = convert_to_geojson(data)
    with open(c + ".geojson", "w", encoding="utf-8") as f:
        json.dump(converted, f, indent=4)

In [ ]:
with open("Population density.json", "r", encoding="utf-8") as f:
    data = json.load(f)
converted = convert_to_geojson(data)
with open("Population density.json", "w", encoding="utf-8") as f:
        json.dump(converted, f, indent=4)

In [4]:
for c in cities:
    with open(c + ".geojson", "r") as f:
        data = json.load(f)
    for feat in data["features"]:
        props = feat["properties"]
        props["city"] = c
    with open(c + ".geojson", "w") as f:
        json.dump(data, f, indent=4)

In [ ]:
cities = [
    "Rome",
    "Milan",
    "Munchen",
    "Berlin",
    "Hamburg",
    "Prague",
    "Budapest",
    "Paris",
    "Marseille",
    "Lille",
    "Lyon",
    "Barcelona",
    "Madrid",
    "Sevilla",
    "Bilbao",
    "Lisbon",
    "Porto",
    "Istanbul",
    "Stockholm",
    "Oslo",
    "Copenhagen",
    "Helsinki",
    "Amsterdam",
    "Rotterdam",
    "Bruxelles",
    "Monaco",
    "Sofia",
    "London",
    "Manchester",
    "Glasgow",
    "Dublin",
    "Bucarest",
    "Riga",
    "Tallin",
    "Vilnius",
    "Ljubljana",
    "Wien",
    "Berna",
    "Zurich",
    "Belgrade",
    "Warsaw"
]

In [3]:
for c in cities:
    query = f"""
    [out:json];
    relation["boundary"="administrative"]
    ["name"="{c}"]
    ["admin_level"="8"];
    out geom;
    """
    print(query)


    [out:json];
    relation["boundary"="administrative"]
    ["name"="Rome"]
    ["admin_level"="8"];
    out geom;
    

    [out:json];
    relation["boundary"="administrative"]
    ["name"="Milan"]
    ["admin_level"="8"];
    out geom;
    

    [out:json];
    relation["boundary"="administrative"]
    ["name"="Munchen"]
    ["admin_level"="8"];
    out geom;
    

    [out:json];
    relation["boundary"="administrative"]
    ["name"="Berlin"]
    ["admin_level"="8"];
    out geom;
    

    [out:json];
    relation["boundary"="administrative"]
    ["name"="Hamburg"]
    ["admin_level"="8"];
    out geom;
    

    [out:json];
    relation["boundary"="administrative"]
    ["name"="Prague"]
    ["admin_level"="8"];
    out geom;
    

    [out:json];
    relation["boundary"="administrative"]
    ["name"="Budapest"]
    ["admin_level"="8"];
    out geom;
    

    [out:json];
    relation["boundary"="administrative"]
    ["name"="Paris"]
    ["admin_level"="8"];
    out geom;
    


In [ ]:
import json
from shapely.geometry import shape

#per città

for c in cities:
    with open(c + "_borders.geojson", "r", encoding="utf-8") as f:
        borders_data = json.load(f)
    borders = shape(borders_data["features"][0]["geometry"])
    with open( c + ".geojson", "r", encoding="utf-8") as f:
        data = json.load(f)
    filtered_features = []
    for feat in data["features"]:
        hexagon = shape(feat["geometry"])
    if borders.contains(hexagon):
        filtered_features.append(feat)
    output = {
        "type": "FeatureCollection",
        "features": filtered_features
    }
    with open(c + "_filtered.geojson", "w", encoding="utf-8") as f:
        json.dump(output, f, indent=4)

In [ ]:
import json
from shapely.geometry import shape


for c in cities:
    with open(c + "_borders.geojson", "r", encoding="utf-8") as f:
        borders_data = json.load(f)
    borders = shape(borders_data["features"][0]["geometry"])
    with open("Population density.geojson", "r", encoding="utf-8") as f:
        data = json.load(f)
    filtered_features = []
    for feat in data["features"]:
        hexagon = shape(feat["geometry"])
    if borders.contains(hexagon):
        filtered_features.append(feat)
        props = feat["properties"]
        props["city"] = c
    output = {
        "type": "FeatureCollection",
        "features": filtered_features
    }
    with open(c + "_filtered.geojson", "w", encoding="utf-8") as f:
        json.dump(output, f, indent=4)


In [ ]:
with open("Rome_filtered.geojson", "r") as f:
    data = json.load(f)
cities.remove("Rome")

for c in cities:
    with open(c + "_filtered.geojson", "r") as f:
        new_data = json.load(f)

        data["features"].extend(new_data["features"])

with open("Cities population.json", "w") as f:
    json.dump(data, f, indent=4)

In [ ]:
with open("Population.geojson", "r") as f:
    data = json.load(f)

for c in cities:
    with open(c + "_borders.geojson", "r", encoding="utf-8") as f:
        borders_data = json.load(f)
    borders = shape(borders_data["features"][0]["geometry"])
    with open( "Population.json", "r", encoding="utf-8") as f:
        data = json.load(f)
    filtered_features = []
    for feat in data["features"]:
        hexagon = shape(feat["geometry"])
        props = feat["properties"]
        props["city"] = c
    if borders.contains(hexagon):
        filtered_features.append(feat)
    output = {
        "type": "FeatureCollection",
        "features": filtered_features
    }
    with open("Population_filtered.geojson", "w", encoding="utf-8") as f:
        json.dump(output, f, indent=4)